In [2]:


import os, gc, math, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [3]:


SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

NUM_GPUS = torch.cuda.device_count()
print(f"GPUs available: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Primary device: {DEVICE}")

GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Primary device: cuda


In [4]:


MODEL_NAME = "jjzha/jobbert-base-cased"
OUTPUT_DIR = "/kaggle/working/jobbert_output"
LOGGING_DIR = "/kaggle/working/jobbert_logs"


TRAIN_CSV = (
    "/kaggle/input/datasets/danielantoniudumitru/"
    "talentclef-subtaska/Training/normalization/"
    "normalized_job_applicant_dataset.csv"
)
DEV_BASE = "/kaggle/input/datasets/danielantoniudumitru/talentclef-subtaska/Development/en"
CORPUS_DIR = os.path.join(DEV_BASE, "corpus")
QUERY_DIR = os.path.join(DEV_BASE, "queries")
QRELS_FILE = os.path.join(DEV_BASE, "qrels.tsv")


MAX_LEN = 512
TRAIN_BATCH = 8
EVAL_BATCH = 16
GRAD_ACCUM = 4
NUM_EPOCHS = 4
LR = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

In [6]:


text_a_col = "query_text"
text_b_col = "corpus_text"
label_col = "relevance"

print(f"Text A: {text_a_col} | Text B: {text_b_col} | Label: {label_col}")

Text A: query_text | Text B: corpus_text | Label: relevance


In [8]:


print(f"\nLoading tokenizer: {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


Loading tokenizer: jjzha/jobbert-base-cased …


config.json:   0%|          | 0.00/603 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:


def read_text_file(folder, file_id):
    path = os.path.join(folder, str(file_id))
    if not os.path.exists(path):
        return ""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read().strip()

In [10]:


print("\nLoading qrels …")
qrels = pd.read_csv(
    QRELS_FILE, sep="\t", header=None,
    names=["q_id", "iter", "c_id", "relevance"]
)
print(f"qrels shape: {qrels.shape}")
print(qrels.head())


Loading qrels …
qrels shape: (472, 4)
    q_id  iter   c_id  relevance
0  36044     0  13884          1
1  39060     0   9516          1
2  39060     0  12097          1
3  32447     0  13882          1
4  39060     0   6533          1


In [11]:


print("\nBuilding (query, corpus) pairs …")
all_q_ids = qrels["q_id"].unique().tolist()
all_c_ids = qrels["c_id"].unique().tolist()
positive_set = set(zip(qrels["q_id"], qrels["c_id"]))

dev_records = []

for _, row in tqdm(qrels.iterrows(), total=len(qrels), desc="Reading positive pairs"):
    q_text = read_text_file(QUERY_DIR,  row["q_id"])
    c_text = read_text_file(CORPUS_DIR, row["c_id"])
    dev_records.append({
        "q_id": row["q_id"],
        "c_id": row["c_id"],
        "query_text": q_text,
        "corpus_text": c_text,
        "relevance": 1
    })

neg_pairs = [
    (q, c)
    for q in all_q_ids
    for c in all_c_ids
    if (q, c) not in positive_set
]

for q_id, c_id in tqdm(neg_pairs, desc="Reading negative pairs"):
    q_text = read_text_file(QUERY_DIR,  q_id)
    c_text = read_text_file(CORPUS_DIR, c_id)
    dev_records.append({
        "q_id": q_id,
        "c_id": c_id,
        "query_text": q_text,
        "corpus_text": c_text,
        "relevance": 0
    })

df_all = pd.DataFrame(dev_records)
print(f"Total pairs: {len(df_all)}")
print(f"Positives: {(df_all['relevance']==1).sum()}")
print(f"Negatives: {(df_all['relevance']==0).sum()}")

df_train_split, df_temp = train_test_split(
    df_all, test_size=0.30, random_state=SEED, stratify=df_all["relevance"]
)
df_val_split, df_eval_split = train_test_split(
    df_temp, test_size=(2/3), random_state=SEED, stratify=df_temp["relevance"]
)

print(f"Train: {len(df_train_split)}")
print(f"Val: {len(df_val_split)}")
print(f"Eval: {len(df_eval_split)}")


Building (query, corpus) pairs …


Reading positive pairs:   0%|          | 0/472 [00:00<?, ?it/s]

Reading negative pairs:   0%|          | 0/3728 [00:00<?, ?it/s]

Total pairs: 4200
Positives: 472
Negatives: 3728
Train: 2940
Val: 420
Eval: 840


In [12]:


class PairDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len,
                 col_a, col_b, label_col):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.col_a = col_a
        self.col_b = col_b
        self.label_col = label_col

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        def clean_text(t):
            return " ".join(str(t).split()[:256]) if pd.notna(t) else ""

        text_a = clean_text(row[self.col_a]) if pd.notna(row[self.col_a]) else ""
        text_b = clean_text(row[self.col_b]) if pd.notna(row[self.col_b]) else ""
        
        encoding = self.tokenizer(
            text_a,
            text_b,
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_token_type_ids=True
        )
        encoding["labels"] = int(row[self.label_col])
        return encoding


train_dataset = PairDataset(df_train_split, tokenizer, MAX_LEN,
                             text_a_col, text_b_col, label_col)
val_dataset = PairDataset(df_val_split,  tokenizer, MAX_LEN,
                             text_a_col, text_b_col, label_col)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Sample encoding keys: {list(train_dataset[0].keys())}")

Sample encoding keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [13]:


print(f"\nLoading model: {MODEL_NAME} …")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Loading model: jjzha/jobbert-base-cased …


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: jjzha/jobbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider train

Model parameters: 108,311,810


In [14]:


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "ap": average_precision_score(labels, probs)
    }

In [15]:


steps_per_epoch = math.ceil(len(train_dataset) / (TRAIN_BATCH * max(NUM_GPUS, 1) * GRAD_ACCUM))
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

os.environ["TENSORBOARD_LOGGING_DIR"] = LOGGING_DIR

training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    logging_dir = LOGGING_DIR,


    num_train_epochs = NUM_EPOCHS,
    per_device_train_batch_size = TRAIN_BATCH,
    per_device_eval_batch_size = EVAL_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,


    learning_rate = LR,
    weight_decay = WEIGHT_DECAY,
    warmup_steps = warmup_steps,
    lr_scheduler_type = "cosine",


    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "ap",
    greater_is_better = True,


    report_to = "none",


    fp16 = True,
    dataloader_num_workers = 4,
    seed = SEED,


    disable_tqdm = False
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [16]:


trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
)

In [17]:


print("\n" + "="*60)
print("  Starting JobBERT fine-tuning …")
print("="*60)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining summary:")
print(train_result.metrics)


  Starting JobBERT fine-tuning …


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Ap
1,No log,0.550732,0.888095,0.000000,0.000000,0.000000,0.547969
2,No log,0.360557,0.935714,0.709677,0.717391,0.702128,0.772216
3,No log,0.268657,0.942857,0.777778,0.688525,0.893617,0.881413
4,No log,0.229771,0.945238,0.780952,0.706897,0.872340,0.900566


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training summary:
{'train_runtime': 658.622, 'train_samples_per_second': 17.855, 'train_steps_per_second': 0.279, 'total_flos': 3094186011033600.0, 'train_loss': 1.6993380007536516, 'epoch': 4.0}


In [18]:


class DevPairDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        def clean_text(t):
            return " ".join(str(t).split()[:256]) if pd.notna(t) else ""

        row = self.data.iloc[idx]
        enc = self.tokenizer(
            clean_text(row["query_text"]),
            clean_text(row["corpus_text"]),
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_token_type_ids=True,
        )
        enc["labels"] = int(row["relevance"])
        return enc

dev_dataset = DevPairDataset(df_eval_split, tokenizer, MAX_LEN)

In [19]:


print("\nRunning predictions on dev set …")

raw_preds = trainer.predict(dev_dataset)
logits = raw_preds.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
preds_bin = (probs >= 0.5).astype(int)
labels = raw_preds.label_ids

df_eval_split["pred_prob"] = probs
df_eval_split["pred_label"] = preds_bin


Running predictions on dev set …


In [20]:


def mean_average_precision(df, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    aps = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).reset_index(drop=True)
        rel = grp_sorted[rel_col].values
        if rel.sum() == 0:
            continue
        hits, ap = 0, 0.0
        for rank, r in enumerate(rel, start=1):
            if r == 1:
                hits += 1
                ap += hits / rank
        aps.append(ap / rel.sum())
    return np.mean(aps) if aps else 0.0


def mean_reciprocal_rank(df, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    rrs = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).reset_index(drop=True)
        rel = grp_sorted[rel_col].values
        rr = 0.0
        for rank, r in enumerate(rel, start=1):
            if r == 1:
                rr = 1.0 / rank
                break
        rrs.append(rr)
    return np.mean(rrs) if rrs else 0.0


def precision_at_k(df, k, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    pk_list = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).head(k)
        pk_list.append(grp_sorted[rel_col].mean())
    return np.mean(pk_list) if pk_list else 0.0

In [21]:


acc = accuracy_score(labels, preds_bin)
f1 = f1_score(labels, preds_bin, zero_division=0)
prec = precision_score(labels, preds_bin, zero_division=0)
rec = recall_score(labels, preds_bin, zero_division=0)
ap_overall = average_precision_score(labels, probs)

MAP = mean_average_precision(df_eval_split)
MRR = mean_reciprocal_rank(df_eval_split)
P_at1 = precision_at_k(df_eval_split, k=1)
P_at5 = precision_at_k(df_eval_split, k=5)
P_at10 = precision_at_k(df_eval_split, k=10)

print("\n" + "="*60)
print("  JobBERT — Dev Set Evaluation Results")
print("="*60)
print(f"  Accuracy: {acc:.4f}")
print(f"  F1: {f1:.4f}")
print(f"  Precision (binary): {prec:.4f}")
print(f"  Recall: {rec:.4f}")
print(f"  Average Precision: {ap_overall:.4f}")
print("─"*60)
print(f"  MAP(Mean Avg Precision): {MAP:.4f}")
print(f"  MRR(Mean Recip Rank): {MRR:.4f}")
print(f"  P@1: {P_at1:.4f}")
print(f"  P@5: {P_at5:.4f}")
print(f"  P@10: {P_at10:.4f}")
print("="*60)


  JobBERT — Dev Set Evaluation Results
  Accuracy: 0.9667
  F1: 0.8614
  Precision (binary): 0.8131
  Recall: 0.9158
  Average Precision: 0.9294
────────────────────────────────────────────────────────────
  MAP(Mean Avg Precision): 0.9044
  MRR(Mean Recip Rank): 0.9500
  P@1: 0.9000
  P@5: 0.8400
  P@10: 0.6400


In [22]:


df_eval_split.to_csv("/kaggle/working/jobbert_dev_predictions.csv", index=False)
print("\nPredictions saved to /kaggle/working/jobbert_dev_predictions.csv")


Predictions saved to /kaggle/working/jobbert_dev_predictions.csv
